# N-Queens Problem — Exhaustive Depth-First Search (DFS)

| | |
|---|---|
| **Name** | Ramya Mercy Rajan |
| **Course** | MSc Software Engineering |
| **University** | University of Europe for Applied Sciences |
| **Professor** | Raja Hashim Ali |


In [4]:
import time
import psutil
import os
import sys

sys.setrecursionlimit(200_000)

KNOWN_SOLUTIONS = {
    1: 1, 2: 0, 3: 0, 4: 2, 5: 10, 6: 4, 7: 40, 8: 92,
    9: 352, 10: 724, 11: 2680, 12: 14200, 13: 73712,
    14: 365596, 15: 2279184, 16: 14772512, 17: 95815104,
    18: 666090624, 19: 4968057848, 20: 39029188884,
    25: 2207893435808352,
}

THEORETICAL_TIME = {
    13: "~2 min",    14: "~10 min",   15: "~1 hr",
    16: "~6 hrs",    17: "~36 hrs",   18: "~9 days",
    19: "~54 days",  20: "~1 year",   25: "~centuries",
    30: "astronomical",
}

In [6]:
def measure_memory():
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 * 1024)

In [8]:
def is_safe(board, row, col):
    for r in range(row):
        c = board[r]
        if c == col or abs(c - col) == abs(r - row):
            return False
    return True

In [10]:
def dfs_solve(N, timeout=10, find_all=True):
    start      = time.time()
    mem_before = measure_memory()

    board       = [-1] * N
    solutions   = []
    timeout_hit = [False]          

    def dfs(row):
       
        if time.time() - start > timeout:
            timeout_hit[0] = True
            return

        
        if row == N:
            solutions.append(board.copy())
            return

        
        for col in range(N):
            if is_safe(board, row, col):
                board[row] = col
                dfs(row + 1)
                if timeout_hit[0]:
                    return
                if not find_all and solutions:
                    return      # stop after first solution

    
    try:
        dfs(0)
    except RecursionError:
        timeout_hit[0] = True   

    elapsed   = time.time() - start
    mem_after = measure_memory()

    return {
        "solutions_found": len(solutions),
        "first_solution":  solutions[0] if solutions else None,
        "time_sec":        round(elapsed, 4),
        "memory_MB":       round(mem_after - mem_before, 4),
        "timeout_hit":     timeout_hit[0],
        "known_total":     KNOWN_SOLUTIONS.get(N, "unknown"),
        "projected_time":  THEORETICAL_TIME.get(N, None),
    }

In [12]:
def print_board(board):
    N = len(board)
    print()
    for row in range(N):
        line = ""
        for col in range(N):
            line += " Q " if board[row] == col else " . "
        print(line)
    print()

In [27]:
def run_experiments():
    # Testing different board sizes
    # Larger values of N take much more time because DFS checks many possibilities
    test_sizes = list(range(4, 16)) + [20, 25, 30]

    # Maximum time allowed for each board size
    TIMEOUT = 10

    print("=" * 68)
    print("       N-Queens Problem using DFS")
    print(f"       Timeout for each test : {TIMEOUT} seconds")
    print("=" * 68)

    header = (f"{'N':>4}  {'Found':>10}  {'Known Total':>14}  "
              f"{'Time (s)':>9}  Status")

    print(f"\n{header}")
    print("-" * 68)

    for N in test_sizes:

       
        result = dfs_solve(N, timeout=TIMEOUT, find_all=True)

        found = result["solutions_found"]
        known = result["known_total"]
        elapsed = result["time_sec"]
        t_hit = result["timeout_hit"]
        proj = result["projected_time"]

        
        if not t_hit and isinstance(known, int) and found == known:
            status = "COMPLETE"

        elif t_hit:
            proj_str = f" (estimated full runtime: {proj})" if proj else ""
            status = f"TIMEOUT{proj_str}"

        else:
            status = "PARTIAL"

   
        print(f"{N:>4}  {str(found):>10}  {str(known):>14}  "
              f"{elapsed:>9.4f}  {status}")

       
        if N == 12:
            print("-" * 68)
            print(" N = 12 is the practical limit for DFS in this experiment")
            print(" Beyond this point the runtime increases very quickly")
            print("-" * 68)

       
        if not t_hit and result["first_solution"] and N <= 8:
            print(f" First solution: {result['first_solution']}")
            print_board(result["first_solution"])

    
    print()
    print("=" * 68)

run_experiments()

       N-Queens Problem using DFS
       Timeout for each test : 10 seconds

   N       Found     Known Total   Time (s)  Status
--------------------------------------------------------------------
   4           2               2     0.0000  COMPLETE
 First solution: [1, 3, 0, 2]

 .  Q  .  . 
 .  .  .  Q 
 Q  .  .  . 
 .  .  Q  . 

   5          10              10     0.0000  COMPLETE
 First solution: [0, 2, 4, 1, 3]

 Q  .  .  .  . 
 .  .  Q  .  . 
 .  .  .  .  Q 
 .  Q  .  .  . 
 .  .  .  Q  . 

   6           4               4     0.0010  COMPLETE
 First solution: [1, 3, 5, 0, 2, 4]

 .  Q  .  .  .  . 
 .  .  .  Q  .  . 
 .  .  .  .  .  Q 
 Q  .  .  .  .  . 
 .  .  Q  .  .  . 
 .  .  .  .  Q  . 

   7          40              40     0.0034  COMPLETE
 First solution: [0, 2, 4, 6, 1, 3, 5]

 Q  .  .  .  .  .  . 
 .  .  Q  .  .  .  . 
 .  .  .  .  Q  .  . 
 .  .  .  .  .  .  Q 
 .  Q  .  .  .  .  . 
 .  .  .  Q  .  .  . 
 .  .  .  .  .  Q  . 

   8          92              92     0.0